# 03 — EDA and Leakage Audit

Exploratory analysis of the cleaned dataset followed by a formal data-leakage audit that confirms leaky features cause an AUC jump > 0.05.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

os.makedirs('../figures', exist_ok=True)


## 1 · Load Cleaned Data

In [ ]:
df = pd.read_parquet('../data/transit_events_clean.parquet')
df['y_primary'] = (df['next_hour_avg_delay'] >= 5.0).astype(int)
print(f"Shape: {df.shape}  |  Positive rate: {df['y_primary'].mean():.3f}")


## 2 · Delay Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['delay_minutes'].clip(-5, 30).hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('delay_minutes (clipped)')
axes[0].set_xlabel('Minutes')

df['next_hour_avg_delay'].clip(0, 30).hist(bins=60, ax=axes[1], color='coral', edgecolor='white')
axes[1].axvline(5.0, color='black', linestyle='--', label='threshold = 5 min')
axes[1].set_title('next_hour_avg_delay (target variable)')
axes[1].set_xlabel('Minutes')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/03_delay_distributions.png', bbox_inches='tight')
plt.show()


## 3 · Positive Rate by Hour, Station, Route

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# By hour
hr = df.groupby('hour_of_day')['y_primary'].mean()
hr.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Positive Rate by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('P(delay ≥ 5 min)')
axes[0].tick_params(axis='x', rotation=45)

# By station (top 20)
st = df.groupby('station_id')['y_primary'].mean().sort_values(ascending=False).head(20)
st.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Positive Rate — Top 20 Stations')
axes[1].set_xlabel('Station')
axes[1].tick_params(axis='x', rotation=45)

# By route
rt = df.groupby('route_id')['y_primary'].mean().sort_values(ascending=False)
rt.plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Positive Rate by Route')
axes[2].set_xlabel('Route')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/03_positive_rates.png', bbox_inches='tight')
plt.show()


## 4 · Target Rate by Weather Regime

In [ ]:
weather_rate = df.groupby('weather_regime')['y_primary'].agg(['mean', 'count'])
weather_rate.columns = ['positive_rate', 'n_events']
weather_rate['positive_rate'] = weather_rate['positive_rate'].round(4)
print("Overall positive rate:", round(df['y_primary'].mean(), 4))
print()
print(weather_rate.sort_values('positive_rate', ascending=False))


## 5 · Leakage Audit

We train a simple logistic regression on **safe** lag features only, record the AUC, then add the deliberately leaky column `current_hour_avg_delay` and confirm AUC jumps by > 0.05.

In [ ]:
import targets

modeling_df = targets.build_modeling_table(df)
print(f"Modeling table shape: {modeling_df.shape}")
modeling_df.head(3)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Strict temporal split — first 60 % train, last 40 % test
modeling_df = modeling_df.sort_values('timestamp').reset_index(drop=True)
n = len(modeling_df)
train_end = int(n * 0.60)

SAFE_FEATURES = [
    c for c in modeling_df.columns
    if c.startswith('lag_') or c.startswith('roll_')
]
print(f"Safe features ({len(SAFE_FEATURES)}): {SAFE_FEATURES[:8]} ...")

X_train = modeling_df.iloc[:train_end][SAFE_FEATURES].fillna(0)
X_test  = modeling_df.iloc[train_end:][SAFE_FEATURES].fillna(0)
y_train = modeling_df.iloc[:train_end]['y_primary']
y_test  = modeling_df.iloc[train_end:]['y_primary']

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr_safe = LogisticRegression(max_iter=500, random_state=42)
lr_safe.fit(X_train_s, y_train)
auc_safe = roc_auc_score(y_test, lr_safe.predict_proba(X_test_s)[:, 1])
print(f"AUC (safe features only): {auc_safe:.4f}")


In [ ]:
# Add leaky feature
LEAKY_COL = 'current_hour_avg_delay'
assert LEAKY_COL in modeling_df.columns, f"Column '{LEAKY_COL}' not found in modeling table."

LEAKY_FEATURES = SAFE_FEATURES + [LEAKY_COL]
X_train_l = modeling_df.iloc[:train_end][LEAKY_FEATURES].fillna(0)
X_test_l  = modeling_df.iloc[train_end:][LEAKY_FEATURES].fillna(0)

scaler_l = StandardScaler()
X_train_ls = scaler_l.fit_transform(X_train_l)
X_test_ls  = scaler_l.transform(X_test_l)

lr_leaky = LogisticRegression(max_iter=500, random_state=42)
lr_leaky.fit(X_train_ls, y_train)
auc_leaky = roc_auc_score(y_test, lr_leaky.predict_proba(X_test_ls)[:, 1])
print(f"AUC (with leaky feature) : {auc_leaky:.4f}")


In [ ]:
delta = auc_leaky - auc_safe
print(f"AUC delta (leaky − safe): {delta:+.4f}")
assert delta > 0.05, (
    f"Leakage audit FAILED — AUC jump ({delta:.4f}) <= 0.05. "
    "Check that current_hour_avg_delay is truly leaky."
)
print("Leakage audit PASSED — leaky feature causes AUC jump > 0.05 ✓")

import pandas as pd
audit_table = pd.DataFrame({
    'model'         : ['LogReg (safe)', 'LogReg (safe + leaky)'],
    'features_used' : [len(SAFE_FEATURES), len(LEAKY_FEATURES)],
    'test_AUC'      : [round(auc_safe, 4), round(auc_leaky, 4)],
    'delta'         : ['—', f'+{delta:.4f}'],
})
print(audit_table.to_string(index=False))


## 6 · Save Modeling Table

In [ ]:
out_path = '../data/modeling_table.parquet'
modeling_df.to_parquet(out_path, index=False)
print(f"Saved → {out_path}  |  shape: {modeling_df.shape}")
